# 0. Libraries import

In [1]:
from utils.cif_utils import get_material_and_cif
from utils.run_calculations_utils import *

/home/edu/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


# 1. Material selection

In [5]:
# Buscamos materiales candidatos
elementos = ["Au", "Li"]
composicion = 2
e_above_hull = 0.1 # Default: 0.05

materiales = get_material_and_cif(e_above_hull, composition=composicion, elements=elementos, verbose=True)

Index	Material	Material ID	Energy above hull	Number of sites in primitive cell
0	LiAu      	mp-1206794	0.00000			2
1	Li3Au     	mp-11247  	0.00000			4
2	LiAu3     	mp-11248  	0.00000			4
3	Li15Au4   	mp-567395 	0.00000			38
4	Li3Au     	mp-1185200	0.00417			4
5	LiAu3     	mp-975909 	0.00424			4
6	LiAu3     	mp-1185333	0.00527			8

Number of materials: 7


In [6]:
### Select material
n = 6
folder_path = 'resultados_calculos'
version = "" #Default = ""

material, material_id, e_hull, nsites = select_material(materiales, n)
material_id=material_id+version
print("Version:", material_id)

Material Elegido:
LiAu3, mp-1185333
Version: mp-1185333


# 2. Slab objects creation

In [5]:
### Create slab objects for the selected material

# Parameters for slabs creation. Must ensure that a slab is generated and has, at least, 7 layers
    # (so each slab has at least 6 completed layers).
min_slab = 20.0 # default: 10.0
if_exists_do = "nothing" # default: "nothing"

# Create slabs and save them in .joblib files
create_slabs_per_miller_index(material, material_id, min_slab=min_slab, if_exists_do=if_exists_do)

Ag
(1, 0, 0):	El slab ya existe	Cantidad de átomos: 10,	Cantidad de capas: 10
(1, 1, 0):	El slab ya existe	Cantidad de átomos: 14,	Cantidad de capas: 14
(1, 1, 1):	El slab ya existe	Cantidad de átomos: 9,	Cantidad de capas: 9


In [68]:
"""
This cell is used to cut the first num_first_layers_to_cut layers. For safety, only is ran when there is a value in
num_first_layers_to_cut non-zero.
"""
num_first_layers_to_cut = {
    '100' : 0,
    '110' : 0,
    '111' : 0
}
new_version_value = "_v2" # Value that will form the new id of the surface (material_id+version).

cut_first_layers(material, material_id, num_first_layers_to_cut, new_version_value)

# 3. .in files generations to cleaned surfaces

In [5]:
replace_in = False

# ¡Ojo acá! Para arreglar en el código. La carta 'outdir' está deprecada, se debe usar la variable de entorno
# ESPRESSO_TMPDIR
cleaned_in_files_creation(material, material_id, e_hull, folder_path, replace_in=replace_in)

LiAu3_mp-1185333
100


/home/edu/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


¿Is the surface stteped? (y/n): n

You entered: 'n'.




/home/edu/Facultad/Trabajo_Final/calculos_definitivos/utils/run_helpers.py:158: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_history = pd.concat([df_history, new_row], ignore_index=True)


110


...

/home/edu/Facultad/Trabajo_Final/calculos_definitivos/utils/run_helpers.py:158: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_history = pd.concat([df_history, new_row], ignore_index=True)



You entered: 'n'.


111


/home/edu/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


¿Is the surface stteped? (y/n): n

You entered: 'n'.




/home/edu/Facultad/Trabajo_Final/calculos_definitivos/utils/run_helpers.py:158: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_history = pd.concat([df_history, new_row], ignore_index=True)


# 4. Actualización de base de datos luego de cálculos de superficies limpias |
# Creamos archivos .in para cada sitio de adsorción

In [7]:
ads_in_files_creation(material, material_id, folder_path)

LiAu3_mp-1185333

-----
100
-----
El archivo LiAu3_mp-1185333_100_relax.out no existe.
-----
110
-----
El archivo LiAu3_mp-1185333_110_relax.out no existe.
-----
111
-----
El archivo LiAu3_mp-1185333_111_relax.out no existe.


FileNotFoundError: [Errno 2] No such file or directory: 'resultados_calculos/LiAu3_mp-1185333'

# 5. Actualización final de la bases de datos con los cálculos completos | Actualización de base de Datos de Energías de Adsorción


In [98]:
complete_history_with_ads_calcs(folder_path, material, material_id)

if_energies_are_in_base_do = 'replace' # defualt = 'nothing'

update_energy_adsorption_database(material, material_id, folder_path, if_energies_are_in_base_do=if_energies_are_in_base_do)

# 6. Impresión de cantidad de sitios de adsorpción calculados por combinación de elementos químicos

In [11]:
adsorption_energies_count(elements=['Mg'])

site type            bridge  hollow  ontop
element_combination                       
(Ag, Mg)                 15       4      8
(Al, Mg)                 13       7      9
(Au, Mg)                 14       9     10
(B, Mg)                  11       4      6
(Cd, Mg)                 15       8      9
(Co, Mg)                  4       2      2
(Cu, Mg)                 17      11     11
(Ga, Mg)                 11       8      9
(Ge, Mg)                 11       3      5
(Hg, Mg)                 14       9     10
(Ir, Mg)                 15       3      7
(Li, Mg)                 11       9      9
(Mg, Mn)                  5       3      5
(Mg, Ni)                 13      10      7
(Mg, Pd)                 13      12      9
(Mg, Pt)                 13       7      9
(Mg, Rh)                  4       4      4
(Mg, Si)                 15       7      8
(Mg, Sn)                 15       8     10
(Mg, Ti)                 10       8      8
(Mg, Y)                  13       7      9
(Mg, Zn)   

# Consultar base de datos de historial

In [12]:
mat_summary = consult_history_db('Ag')
pd.set_option('display.max_columns', None)
display(mat_summary)

,material,material-id,e_hull,indice_de_miller,capas,nsites,altura_slab,a,b,c,alpha,beta,gamma,escalonada,exito_limpia,sitios_bridge,sitios_hollow,sitios_ontop,sitios_totales,bridge_enviados,hollow_enviados,ontop_enviados,bridge_exitosos,hollow_exitosos,ontop_exitosos,prob_convergencia,prob_tiempo,errores
225,Ag,mp-8566,0.000000,100,6,16,9.244084,2.911131,9.463709,29.244084,90.000000,90.000000,90.000000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
226,Ag,mp-8566,0.000000,110,6,24,7.598590,5.042228,9.463709,27.598590,90.000000,94.715004,90.000000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
227,Ag,mp-8566,0.000000,111,6,21,7.598590,5.042228,9.901337,27.598590,99.638976,94.715004,75.248718,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0
228,Ag,mp-124,0.002127,100,6,6,10.260891,2.902218,2.902218,30.260891,90.000000,90.000000,90.000000,False,True,2.0,0.0,1.0,3.0,2.0,0.0,1.0,2.0,0.0,1.0,0,0,0
229,Ag,mp-124,0.002127,110,6,6,7.255546,2.902218,4.104356,27.255546,90.000000,90.000000,90.000000,False,True,3.0,0.0,1.0,4.0,2.0,0.0,1.0,2.0,0.0,1.0,0,0,0
230,Ag,mp-124,0.002127,111,6,6,11.917695,2.902218,2.902218,31.917695,87.496225,87.496225,60.000000,False,True,2.0,2.0,1.0,5.0,2.0,2.0,1.0,2.0,2.0,1.0,0,0,0
231,Ag,mp-10597,0.007669,100,6,12,13.497522,2.922299,4.701662,33.497522,90.000000,90.000000,90.000000,False,True,3.0,2.0,2.0,7.0,0.0,2.0,0.0,0.0,2.0,0.0,0,0,0
232,Ag,mp-10597,0.007669,110,6,12,7.489054,4.701662,5.061571,27.489054,94.715004,90.000000,90.000000,False,True,4.0,2.0,2.0,8.0,0.0,2.0,0.0,0.0,2.0,0.0,0,0,0
233,Ag,mp-10597,0.007669,111,6,12,7.583841,5.061571,5.535834,27.583841,92.260041,94.508939,62.795728,False,True,6.0,2.0,2.0,10.0,0.0,2.0,0.0,0.0,2.0,0.0,0,0,0
234,Ag,mp-989737,0.010382,100,6,18,4.252321,2.946094,21.613950,24.252321,90.000000,90.000000,90.000000,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0


In [13]:
consult_energies_database(material, material_id)

,material,material id,miller index,site type,site number,Adsorption energy
